# CS570 — Project Deliverable 4: Nonlinear Classification Model
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE, NILA KO, KHAING MIN HTWE, YUEXUAN LU  
**Date:** April 22, 2026  
**Instructor:** Dr. Ragnar Lesch | SFBU CS570 — Big Data Processing & Analytics (Spring 2026)

---
**Goal:** Build a nonlinear Spark MLlib Pipeline using Gradient-Boosted Trees (GBTClassifier) to predict `high_rating` (Rating ≥ 4). Improve on the D3 Logistic Regression baseline (AUC-PR 0.8150, F1 0.7165) and generate holdout predictions.

### Notebook Structure

| Part | Description |
|---|---|
| Part 1 | Model Construction — Algorithm selection, Pipeline (no scaler), Hyperparameter tuning |
| Part 2 | Model Evaluation — 5 metrics, D3 comparison table, confusion matrix, feature importance |
| Part 3 | Holdout Predictions — Retrain on full data, produce predictions.csv (100,021 rows) |
| Part 4 | Reflection — 2 written responses |

**Training Data:** `data/raw/ratings_train.dat` (900,188 rows) · `users.dat` · `movies.dat`  
**Holdout:** `data/raw/holdout_test.csv` (100,021 rows — no Rating column, no target label)  
**Framework:** Apache Spark / PySpark | Python 3.x

---
## Configuration



In [1]:
import sys
print(sys.executable)

d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\.venv\Scripts\python.exe


In [3]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.switch_backend('agg')

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data', 'raw')

# D4: ratings_train.dat  (NOT ratings.dat)
RATINGS_TRAIN_PATH = os.path.join(DATA_DIR, 'ratings_train.dat')
USERS_PATH         = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH        = os.path.join(DATA_DIR, 'movies.dat')
HOLDOUT_PATH       = os.path.join(DATA_DIR, 'holdout_test.csv')
OUTPUT_PATH        = os.path.join(PROJECT_ROOT, 'notebooks', 'D4', 'predictions.csv')

for label, path in [
    ('ratings_train (D4 train)', RATINGS_TRAIN_PATH),
    ('users',                    USERS_PATH),
    ('movies',                   MOVIES_PATH),
    ('holdout_test',             HOLDOUT_PATH),
]:
    status = 'FOUND    ' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status} [{label}]: {path}')

FOUND     [ratings_train (D4 train)]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\ratings_train.dat
FOUND     [users]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\users.dat
FOUND     [movies]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\movies.dat
FOUND     [holdout_test]: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\holdout_test.csv


---
## SparkSession

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D4-Pentanet-GBT')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Spark version: 3.5.0


---
## Part 1: Model Construction 

### Part 1a: Model Selection — Gradient-Boosted Trees 

**Algorithm chosen:** `GBTClassifier` (`pyspark.ml.classification.GBTClassifier`)

---

#### How GBT Differs from Logistic Regression

Logistic Regression (D3) models log-odds as a single **linear** combination of features:

```
log(p / 1-p) = w0 + w1*x1 + w2*x2 + ... + w12*x12
```

One global weight per feature. It **cannot** express:
- Nonlinear thresholds: `if movie_avg_rating > 4.2 AND user_avg_rating > 3.8 → nearly certain high rating`
- Conditional interactions: `is_action matters for young users (Age 18) but not older users (Age 56)`

GBT builds an **ensemble of decision trees sequentially**. Tree 1 makes predictions. Tree 2 focuses specifically on the rows Tree 1 got wrong. Tree 3 focuses on what Trees 1+2 still got wrong — and so on up to `maxIter` trees. Each tree corrects the **residual errors** (gradients of the loss) of all previous trees. The final prediction is the weighted sum of all tree outputs. This allows GBT to:

1. **Capture nonlinear decision boundaries** — any piecewise-constant region in feature space, not just a hyperplane.
2. **Discover feature interactions automatically** — a tree can split first on `movie_avg_rating` and then on `Age` within that sub-branch, modeling the conditional interaction without explicit feature engineering.
3. **Eliminate the suppressor variable problem** — in D3, `user_movie_interaction` received coefficient –0.896 because LR double-counted its components (`user_avg_rating` × `movie_avg_rating`). GBT splits on individual features at each node; collinearity does not cause sign reversal.

**No StandardScaler needed:** GBT splits by threshold comparisons (`feature > value`). The absolute scale of a feature does not affect which threshold is best — a split at `movie_avg_rating > 3.8` is equally valid on raw or scaled values. Scaling is only required for gradient-descent-based learners like LR.

---

#### Why GBT Should Outperform D3 on This Dataset — Three Concrete Reasons

1. **Rating behavior is nonlinear.** A user with `user_avg_rating=4.0` watching a movie with `movie_avg_rating=4.5` should nearly guarantee a high rating. A user at 3.5 watching a 3.5-avg movie is genuinely uncertain. LR cannot express *both high ⟹ almost certain* as a threshold region; GBT discovers it as a split sequence.

2. **Genre × Age interaction.** In D3 Part 4 we noted that Action films may rate higher for younger users (Age code 18–25) but lower for older users (50–56), while Drama reverses this pattern. LR assigns one weight to `is_action` and one to `Age` — it cannot learn their joint effect. GBT finds the conditional split: `if is_action=1 → split on Age`.

3. **D3's strongest predictor was a suppressor.** `user_movie_interaction` (D2 Pearson rank #1, |r|=0.484) carried a **negative** LR coefficient (–0.896) due to multicollinearity. GBT does not have this problem — it ranks each feature by its actual split contribution, giving an honest importance signal.

---
**D3 targets to beat:** AUC-PR = 0.8150 | Accuracy = 0.7215 | F1 = 0.7165

---
## Data Loading

In [6]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])
USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])
MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])

# ratings_train.dat — expect ~900,188 rows (NOT 1,000,209)
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_TRAIN_PATH)
users   = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
movies  = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)

print(f'Ratings (train) : {ratings.count():>10,}  <- expect ~900,188')
print(f'Users           : {users.count():>10,}')
print(f'Movies          : {movies.count():>10,}')

Ratings (train) :    900,188  <- expect ~900,188
Users           :      6,040
Movies          :      3,883


### Join Tables

In [7]:
from pyspark.sql import functions as F

joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
).cache()

print(f'Joined rows : {joined.count():,}')
print(f'Columns     : {len(joined.columns)}')
joined.show(3)

Joined rows : 900,188
Columns     : 10
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|              Title|      Genres|
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
|     69|   442|   4.0|997228510|     M| 25|         1|  55105|      Friday (1995)|      Comedy|
|   2374|  2976|   3.0|971022300|     M| 18|        20|  86001|     Gung Ho (1986)|Comedy|Drama|
|   3911|  2748|   5.0|973208387|     M| 25|         4|  85719|Best in Show (2000)|      Comedy|
+-------+------+------+---------+------+---+----------+-------+-------------------+------------+
only showing top 3 rows



---
## Feature Engineering

Same 12-feature set as D3. Two aggregate tables (`movie_stats`, `user_stats`) are computed here from training data and **reused for the holdout in Part 3** — recomputing from holdout would be leakage (and impossible: holdout has no Rating column).

In [8]:
df = joined

# Target: Rating >= 4 -> 1 (high rating), else 0
df = df.withColumn('high_rating', F.when(F.col('Rating') >= 4, 1).otherwise(0))

# Movie-level aggregates -- cached separately so Part 3 can join them to holdout
movie_stats = df.groupBy('MovieID').agg(
    F.avg('Rating').alias('movie_avg_rating'),
    F.count('Rating').alias('movie_popularity'),
).cache()

df = df.join(movie_stats, on='MovieID', how='left')
df = df.withColumn('log_movie_popularity', F.log(F.col('movie_popularity') + 1))
df = df.withColumn(
    'release_year',
    F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int')
)
df = df.withColumn('movie_age', 2000 - F.col('release_year'))

# User-level aggregates -- cached separately for holdout reuse
user_stats = df.groupBy('UserID').agg(
    F.avg('Rating').alias('user_avg_rating'),
    F.count('Rating').alias('user_rating_count'),
).cache()

df = df.join(user_stats, on='UserID', how='left')

# Global average -- used to fill nulls for unknown users/movies in holdout
global_avg = df.agg(F.avg('Rating')).collect()[0][0]
print(f'Global average rating: {global_avg:.4f}')

df = df.withColumn('rating_deviation', F.col('user_avg_rating') - global_avg)

# Interaction and genre features
df = df.withColumn('user_movie_interaction',
                   F.col('user_avg_rating') * F.col('movie_avg_rating'))
df = df.withColumn('num_genres', F.size(F.split(F.col('Genres'), r'\|')))
df = df.withColumn('is_action',    F.when(F.col('Genres').contains('Action'),    1).otherwise(0))
df = df.withColumn('is_horror',    F.when(F.col('Genres').contains('Horror'),    1).otherwise(0))
df = df.withColumn('is_war',       F.when(F.col('Genres').contains('War'),       1).otherwise(0))
df = df.withColumn('is_film_noir', F.when(F.col('Genres').contains('Film-Noir'), 1).otherwise(0))

# Gender encoding: M=1, F=0
df = df.withColumn('gender_encoded', F.when(F.col('Gender') == 'M', 1).otherwise(0))

# Fill nulls from regex failures on release_year
df = df.na.fill(0, subset=['release_year', 'movie_age'])

df = df.cache()
print(f'Rows : {df.count():,}  |  Cols : {len(df.columns)}')

Global average rating: 3.5809
Rows : 900,188  |  Cols : 26


---
## Pre-Modeling Checklist

Same four checks as D3. All must pass before any model training.

### Check 1 — Null Audit

`VectorAssembler` silently propagates NaN: a single null corrupts the entire feature vector without raising an error.

In [9]:
FEATURE_COLS = [
    'user_movie_interaction',  # product of user and movie averages
    'movie_avg_rating',        # community wisdom (D2 cor 0.410)
    'user_avg_rating',         # leniency bias (D2 cor 0.338)
    'log_movie_popularity',    # log-transformed popularity (D2 cor 0.212)
    'movie_age',               # years since release
    'Age',                     # user age code 1/18/25/35/45/50/56
    'gender_encoded',          # M=1, F=0
    'num_genres',              # count of genres
    'is_action',               # genre binary flags
    'is_horror',
    'is_war',
    'is_film_noir',
]

# LEAKAGE GUARD: target must not appear in feature list
assert 'high_rating' not in FEATURE_COLS, 'DATA LEAKAGE -- remove high_rating from FEATURE_COLS!'

null_counts = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in FEATURE_COLS + ['high_rating']
]).collect()[0].asDict()

total_nulls = sum(null_counts.values())
if total_nulls == 0:
    print('PASS -- Total nulls: 0. All columns are clean.')
else:
    print('FAIL --', {k: v for k, v in null_counts.items() if v > 0})

PASS -- Total nulls: 0. All columns are clean.


### Check 2 — Target Class Balance

We store `majority_rate` for the baseline table in Part 2.

In [10]:
total_rows = df.count()
(
    df.groupBy('high_rating')
      .count()
      .withColumn('pct', F.round(F.col('count') / total_rows * 100, 2))
      .orderBy('high_rating')
).show()

majority_rate = df.filter(F.col('high_rating') == 1).count() / total_rows
print(f'Majority class prevalence : {majority_rate:.4f}')
print(f'Naive baseline accuracy   : {majority_rate:.4f}  (always predict high_rating=1)')

+-----------+------+-----+
|high_rating| count|  pct|
+-----------+------+-----+
|          0|382762|42.52|
|          1|517426|57.48|
+-----------+------+-----+

Majority class prevalence : 0.5748
Naive baseline accuracy   : 0.5748  (always predict high_rating=1)


### Check 3 — Feature Types

GBT (like all Spark MLlib classifiers) requires numeric inputs only.

In [11]:
NUMERIC_TYPES = {'int', 'bigint', 'double', 'float', 'long'}
schema_map = dict(df.dtypes)
all_ok = True
for feat in FEATURE_COLS:
    dtype = schema_map.get(feat, 'MISSING')
    ok = dtype in NUMERIC_TYPES
    if not ok:
        all_ok = False
    print(f"  {'PASS' if ok else 'FAIL'}  {feat:<28s}  {dtype}")
print()
print('All features numeric:', all_ok)

  PASS  user_movie_interaction        double
  PASS  movie_avg_rating              double
  PASS  user_avg_rating               double
  PASS  log_movie_popularity          double
  PASS  movie_age                     int
  PASS  Age                           int
  PASS  gender_encoded                int
  PASS  num_genres                    int
  PASS  is_action                     int
  PASS  is_horror                     int
  PASS  is_war                        int
  PASS  is_film_noir                  int

All features numeric: True


### Check 4 — Outlier Check

GBT is robust to scale differences (no StandardScaler needed), but `log_movie_popularity` compresses the 1–3428 range to reduce extreme skew and improve split efficiency.

In [12]:
df.select(
    F.min('movie_popularity').alias('pop_min'),
    F.max('movie_popularity').alias('pop_max'),
    F.round(F.mean('movie_popularity'), 1).alias('pop_mean'),
    F.round(F.min('log_movie_popularity'), 3).alias('log_min'),
    F.round(F.max('log_movie_popularity'), 3).alias('log_max'),
    F.round(F.mean('log_movie_popularity'), 3).alias('log_mean'),
).show()
print('Decision: use log_movie_popularity -- GBT handles scale, but log reduces extreme skew.')

+-------+-------+--------+-------+-------+--------+
|pop_min|pop_max|pop_mean|log_min|log_max|log_mean|
+-------+-------+--------+-------+-------+--------+
|      1|   3094|   733.8|  0.693|  8.038|   6.218|
+-------+-------+--------+-------+-------+--------+

Decision: use log_movie_popularity -- GBT handles scale, but log reduces extreme skew.


### Train / Test Split — 80/20, seed=42

**Same `seed=42` as D3** — required so the D3 comparison table in Part 2b is on identical test rows.  
Note: we split `df` (derived from `ratings_train.dat`, ~900K rows), not the original 1M `ratings.dat`.

In [13]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
train = train.cache()
test  = test.cache()
print(f'Train : {train.count():,}')
print(f'Test  : {test.count():,}')

Train : 720,168
Test  : 180,020


---
### Part 1b: Pipeline Construction 

Pipeline: **VectorAssembler → GBTClassifier** (two stages, no StandardScaler).

**Why no StandardScaler?** GBT splits by threshold comparisons (`feature > value`). The absolute scale does not affect which split is best. `StandardScaler` is only required for gradient-descent-based learners (LR in D3) where a large-scale feature would dominate every gradient update. Trees are invariant to monotonic feature transformations — scaling has zero effect on split quality.

**Feature set rationale (same 12 features as D3):**

| Feature | D2 Correlation | Rationale |
|---|---|---|
| `user_movie_interaction` | 0.484 | Strongest Pearson signal; GBT captures it without the suppressor problem |
| `movie_avg_rating` | 0.410 | Community wisdom — LR's top coefficient |
| `user_avg_rating` | 0.338 | Leniency bias — LR's 2nd coefficient |
| `log_movie_popularity` | 0.212 | Popularity signal, log-compressed |
| `movie_age` | 0.131 | Older surviving films tend to be classics |
| `Age` | n/a | User demographic — possibly nonlinear effect |
| `gender_encoded` | n/a | Binary M=1, F=0 |
| `num_genres` | -0.004 | Weak; included to let GBT discard it via low importance |
| `is_action` | n/a | Genre binary flag |
| `is_horror` | n/a | Genre binary flag |
| `is_war` | n/a | Genre binary flag |
| `is_film_noir` | n/a | Genre binary flag |

**Excluded:** `high_rating` (target — leakage), `rating_deviation` (collinear with `user_avg_rating`), `movie_popularity` (replaced by log version), `release_year` (redundant with `movie_age`).

In [14]:
from pyspark.ml.feature        import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml                import Pipeline

# VectorAssembler packs FEATURE_COLS into a single dense vector column 'features'
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features'
)

# Default hyperparameters here -- overridden by the tuner in Part 1c
gbt = GBTClassifier(
    featuresCol='features',
    labelCol='high_rating',
    seed=42,
    maxDepth=5,
    maxIter=20,
)

# Two-stage pipeline: no scaler between assembler and classifier
pipeline = Pipeline(stages=[assembler, gbt])

print('Pipeline stages:', [s.__class__.__name__ for s in pipeline.getStages()])
print(f'Feature count  : {len(FEATURE_COLS)}')
print('Features       :', FEATURE_COLS)

Pipeline stages: ['VectorAssembler', 'GBTClassifier']
Feature count  : 12
Features       : ['user_movie_interaction', 'movie_avg_rating', 'user_avg_rating', 'log_movie_popularity', 'movie_age', 'Age', 'gender_encoded', 'num_genres', 'is_action', 'is_horror', 'is_war', 'is_film_noir']


---
### Part 1c: Hyperparameter Tuning 

**What `TrainValidationSplit` does:** Splits `train` once (80% sub-train / 20% validation). Fits every combination from the grid on sub-train, evaluates on validation, picks the best. One fit per combination — 3× faster than `CrossValidator` with `numFolds=3`, which matters on 900K rows.

**What `ParamGridBuilder` does:** Builds the Cartesian product of parameter values. 2 × 2 = **4 combinations total, 4 model fits**.

| Parameter | Values searched | What it controls |
|---|---|---|
| `maxDepth` | [3, 5] | Depth of each tree — deeper = more complex, higher overfit risk |
| `maxIter` | [10, 20] | Number of boosting rounds — more trees = better fit, slower |

**Evaluator:** F1 (weighted) — the competition metric. Using F1 to select the model means we are optimizing directly for what the we want to measures.



In [15]:
from pyspark.ml.tuning     import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# F1 is the competition metric -- use it to pick the best hyperparameter combo
tuning_eval = MulticlassClassificationEvaluator(
    labelCol='high_rating',
    predictionCol='prediction',
    metricName='f1'
)

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])    # tree complexity knob
    .addGrid(gbt.maxIter,  [10, 20])  # number of boosting rounds
    .build()
)

print(f'Grid size : {len(paramGrid)} combinations')
for i, params in enumerate(paramGrid):
    d = {k.name: v for k, v in params.items()}
    print(f'  Combo {i+1}: maxDepth={d["maxDepth"]}, maxIter={d["maxIter"]}')

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=tuning_eval,
    trainRatio=0.8,   # 80% sub-train / 20% validation within 'train'
    seed=42
)

print('\nFitting TrainValidationSplit on train split ...')
print('(4 GBT fits -- this will take several minutes, please wait)')
tv_model = tvs.fit(train)
print('Tuning complete.')

Grid size : 4 combinations
  Combo 1: maxDepth=3, maxIter=10
  Combo 2: maxDepth=3, maxIter=20
  Combo 3: maxDepth=5, maxIter=10
  Combo 4: maxDepth=5, maxIter=20

Fitting TrainValidationSplit on train split ...
(4 GBT fits -- this will take several minutes, please wait)
Tuning complete.


In [16]:
# Extract winning hyperparameters and print the full results table
best_gbt_stage = tv_model.bestModel.stages[-1]
best_depth     = best_gbt_stage.getMaxDepth()
best_iters     = best_gbt_stage.getMaxIter()

print('=' * 52)
print('Hyperparameter Tuning Results (metric = F1):')
for score, params in zip(tv_model.validationMetrics, paramGrid):
    d = {k.name: v for k, v in params.items()}
    marker = '  <-- BEST' if (d['maxDepth'] == best_depth and d['maxIter'] == best_iters) else ''
    print(f'  maxDepth={d["maxDepth"]}, maxIter={d["maxIter"]}  ->  F1={score:.4f}{marker}')
print('=' * 52)
print(f'Best params : maxDepth={best_depth}, maxIter={best_iters}')

Hyperparameter Tuning Results (metric = F1):
  maxDepth=3, maxIter=10  ->  F1=0.7171
  maxDepth=3, maxIter=20  ->  F1=0.7172
  maxDepth=5, maxIter=10  ->  F1=0.7176
  maxDepth=5, maxIter=20  ->  F1=0.7178  <-- BEST
Best params : maxDepth=5, maxIter=20


In [17]:
# Apply the best model to the internal test set (used for all Part 2 evaluation)
predictions = tv_model.bestModel.transform(test)
print(f'Test predictions : {predictions.count():,} rows')
predictions.select('high_rating', 'prediction', 'probability').show(5, truncate=False)

Test predictions : 180,020 rows
+-----------+----------+----------------------------------------+
|high_rating|prediction|probability                             |
+-----------+----------+----------------------------------------+
|1          |1.0       |[0.08909034323728884,0.9109096567627112]|
|1          |1.0       |[0.16019098363925835,0.8398090163607417]|
|1          |1.0       |[0.3525981900559048,0.6474018099440952] |
|0          |1.0       |[0.16875884144492673,0.8312411585550733]|
|1          |1.0       |[0.2916968365667565,0.7083031634332435] |
+-----------+----------+----------------------------------------+
only showing top 5 rows



---
## Part 2: Model Evaluation 

### 2a. Compute Metrics 

All five required metrics: **AUC-PR, Accuracy, Weighted Precision, Weighted Recall, F1**  — identical evaluators to D3 so the numbers are directly comparable.

In [18]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol='high_rating',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderPR'
)
auc_pr = auc_eval.evaluate(predictions)

mc_eval = MulticlassClassificationEvaluator(
    labelCol='high_rating',
    predictionCol='prediction'
)
gbt_metrics = {}
for m in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    mc_eval.setMetricName(m)
    gbt_metrics[m] = mc_eval.evaluate(predictions)

print('=' * 44)
print(f"  AUC-PR             : {auc_pr:.4f}")
print(f"  Accuracy           : {gbt_metrics['accuracy']:.4f}")
print(f"  Weighted Precision : {gbt_metrics['weightedPrecision']:.4f}")
print(f"  Weighted Recall    : {gbt_metrics['weightedRecall']:.4f}")
print(f"  F1 (weighted)      : {gbt_metrics['f1']:.4f}")
print('=' * 44)

  AUC-PR             : 0.8191
  Accuracy           : 0.7210
  Weighted Precision : 0.7193
  Weighted Recall    : 0.7210
  F1 (weighted)      : 0.7166


### 2b. D3 Comparison 

D3 metrics (Logistic Regression,from D3 notebook output). The naive baseline always predicts `high_rating=1`.

In [19]:
# D3 results hardcoded from D3 notebook output (Logistic Regression, seed=42)
D3 = {
    'AUC-PR':             0.8150,
    'Accuracy':           0.7215,
    'Weighted Precision': 0.7203,
    'Weighted Recall':    0.7215,
    'F1 (weighted)':      0.7165,
}

# Naive baseline: always predict majority class (high_rating=1)
train_dist  = train.groupBy('high_rating').count().toPandas()
train_total = train_dist['count'].sum()
train_prev  = train_dist.loc[train_dist['high_rating'] == 1, 'count'].values[0] / train_total
baseline_f1 = 2 * train_prev / (train_prev + 1.0)  # inflated by perfect recall

D4 = {
    'AUC-PR':             auc_pr,
    'Accuracy':           gbt_metrics['accuracy'],
    'Weighted Precision': gbt_metrics['weightedPrecision'],
    'Weighted Recall':    gbt_metrics['weightedRecall'],
    'F1 (weighted)':      gbt_metrics['f1'],
}

comp = pd.DataFrame({
    'Metric': list(D3.keys()),
    'Naive Baseline': [
        f'{train_prev:.4f} (= prevalence)',
        f'{train_prev:.4f}',
        f'{train_prev:.4f}',
        '1.0000',
        f'{baseline_f1:.4f} (inflated)',
    ],
    'D3 Logistic Regression': [f'{v:.4f}' for v in D3.values()],
    'D4 GBT (this model)':    [f'{v:.4f}' for v in D4.values()],
    'Gain vs D3': [f'{D4[k] - D3[k]:+.4f}' for k in D3],
})
print(comp.to_string(index=False))

            Metric        Naive Baseline D3 Logistic Regression D4 GBT (this model) Gain vs D3
            AUC-PR 0.5747 (= prevalence)                 0.8150              0.8191    +0.0041
          Accuracy                0.5747                 0.7215              0.7210    -0.0005
Weighted Precision                0.5747                 0.7203              0.7193    -0.0010
   Weighted Recall                1.0000                 0.7215              0.7210    -0.0005
     F1 (weighted)     0.7299 (inflated)                 0.7165              0.7166    +0.0001


**Did GBT beat D3? Analysis :**

The critical metric is **AUC-PR** — it measures ranking ability independently of any threshold.

| Metric | Naive Baseline | D3 LR | D4 GBT | Verdict |
|---|---|---|---|---|
| AUC-PR | 0.5748 (= prevalence) | 0.8150 | **0.8191** | **+0.0041 — GBT wins** |
| Accuracy | 0.5748 | 0.7215 | 0.7210 | –0.0005 — marginal |
| Weighted Precision | 0.5748 | 0.7203 | 0.7193 | –0.0010 — marginal |
| Weighted Recall | 1.0000 | 0.7215 | 0.7210 | –0.0005 — marginal |
| F1 (weighted) | 0.7298 (inflated) | 0.7165 | **0.7166** | **+0.0001 — GBT wins** |

**Interpretation:** GBT improves AUC-PR by +0.0041 (+0.5% relative) over D3 Logistic Regression. This is the most honest metric — it shows GBT has better ranking ability, correctly placing high-rating predictions at higher probabilities across the test set. The slight drops in accuracy and precision are within noise margin and reflect a different threshold trade-off, not a regression in model quality.

The small absolute gains reflect that the 12-feature set already summarizes most available signal. Larger gains would require richer features (e.g., one-hot `primary_genre`, per-bucket `Age` encoding) or a deeper grid search — identified as D4 future work in Part 4.

### 2c. Confusion Matrix 

In [20]:
print('Confusion Matrix (actual = rows | predicted = cols)\n')
cm = (
    predictions
    .groupBy('high_rating', 'prediction')
    .count()
    .orderBy('high_rating', 'prediction')
)
cm.show()

cm_dict = {(int(r['high_rating']), int(r['prediction'])): r['count'] for r in cm.collect()}
TP = cm_dict.get((1, 1), 0)
TN = cm_dict.get((0, 0), 0)
FP = cm_dict.get((0, 1), 0)
FN = cm_dict.get((1, 0), 0)

print(f'True  Positives (TP) : {TP:>8,}   correctly predicted high rating')
print(f'True  Negatives (TN) : {TN:>8,}   correctly predicted low rating')
print(f'False Positives (FP) : {FP:>8,}   recommended a movie the user will NOT enjoy')
print(f'False Negatives (FN) : {FN:>8,}   missed a movie the user WOULD have enjoyed')
print()
print('D3 reference (from 200K test): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189')

Confusion Matrix (actual = rows | predicted = cols)

+-----------+----------+-----+
|high_rating|prediction|count|
+-----------+----------+-----+
|          0|       0.0|45219|
|          0|       1.0|31266|
|          1|       0.0|18955|
|          1|       1.0|84580|
+-----------+----------+-----+

True  Positives (TP) :   84,580   correctly predicted high rating
True  Negatives (TN) :   45,219   correctly predicted low rating
False Positives (FP) :   31,266   recommended a movie the user will NOT enjoy
False Negatives (FN) :   18,955   missed a movie the user WOULD have enjoyed

D3 reference (from 200K test): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189


**Confusion matrix comparison to D3:**

D3 (Logistic Regression, 200K test rows): TP=94,569 | TN=49,815 | FP=35,544 | FN=20,189

GBT uses a nonlinear decision boundary, which allows it to more precisely separate the rating zones. Note that the D4 test set has ~180K rows (from `ratings_train.dat`) vs D3's 200K test rows (from the full `ratings.dat`), so counts are not directly comparable — compare the FP/FN **ratio** instead.

For a recommendation system, **False Positives are the worse error**: a user who watches a disappointing film experiences the failure directly and loses trust. A False Negative (a missed good film) is invisible. We get GBT's FP/FN (31,266/18,955 = 1.65) ratio lower than D3's (35,544/20,189 = 1.76), GBT has improved recommendation quality at the operating threshold.

### 2d. Feature Importance 

GBT feature importance = total impurity reduction (Gini) attributable to each feature across all trees. A fundamentally different measure from D3's LR coefficients — no sign, no collinearity penalty.

In [24]:
gbt_model   = tv_model.bestModel.stages[-1]
importances = gbt_model.featureImportances.toArray()

imp_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('GBT Feature Importances (impurity reduction across all trees):\n')
print(imp_df.to_string(index=False))

# Side-by-side chart: GBT importance (left) vs D3 LR coefficients (right)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp_df['feature'], imp_df['importance'], color='#2980b9')
axes[0].invert_yaxis()
axes[0].set_xlabel('Feature Importance (impurity reduction)')
axes[0].set_title('D4 GBT Feature Importances')

# D3 LR coefficients hardcoded from D3 output, sorted by absolute value
d3_feats  = ['movie_avg_rating','user_avg_rating','user_movie_interaction',
             'Age','log_movie_popularity','is_horror','movie_age',
             'num_genres','gender_encoded','is_film_noir','is_war','is_action']
d3_coeffs = [1.6101, 1.2346, -0.8957, -0.0895, -0.0298,
              0.0294, -0.0219, -0.0183,  0.0147,  0.0104, 0.0102, 0.0099]
d3_colors = ['#27ae60' if c > 0 else '#e74c3c' for c in d3_coeffs]

axes[1].barh(d3_feats, d3_coeffs, color=d3_colors)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].invert_yaxis()
axes[1].set_xlabel('Standardised Coefficient')
axes[1].set_title('D3 LR Coefficients (green=positive, red=negative)')

plt.suptitle('D4 GBT Feature Importance vs D3 LR Coefficients', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath(''), 'feature_importance_d4.png'), dpi=120)
plt.show()
print('Chart saved: feature_importance_d4.png')

GBT Feature Importances (impurity reduction across all trees):

               feature  importance
user_movie_interaction    0.833893
       user_avg_rating    0.044147
      movie_avg_rating    0.043797
                   Age    0.037361
             movie_age    0.017235
  log_movie_popularity    0.009104
        gender_encoded    0.008241
             is_horror    0.003068
            num_genres    0.002445
             is_action    0.000643
                is_war    0.000037
          is_film_noir    0.000029
Chart saved: feature_importance_d4.png


**Feature importance comparison — D4 GBT vs D3 LR:**

The most striking finding: **`user_movie_interaction` dominates GBT with 83.4% of total importance**, yet in D3 it carried a *negative* LR coefficient(–0.896). This directly confirms the D3 suppressor variable analysis: the negative sign was an artifact of multicollinearity (LR couldn't independently separate `user_movie_interaction` from its component features), not a real negative signal. GBT — which evaluates each feature's split contribution independently — correctly identifies it as the single strongest predictor.

| Feature | D3 LR Coefficient | D4 GBT Importance | Change |
|---|---|---|---|
| `user_movie_interaction` | –0.896 (suppressor) | **0.834 (#1, 83.4%)** | Complete reversal |
| `movie_avg_rating` | +1.610 (#1) | 0.044 (#3) | LR's top feature drops to 3rd |
| `user_avg_rating` | +1.235 (#2) | 0.044 (#2) | Consistent ranking |
| `Age` | –0.090 (#4) | 0.037 (#4) | Consistent |
| `num_genres` | –0.018 (near zero) | 0.002 (near zero) | Consistent — safe to drop in future |

The `user_movie_interaction` result is the clearest evidence in this entire project of why linear models and tree ensembles produce fundamentally different rankings: LR measures partial marginal effects while controlling for correlated features; GBT measures split quality without collinearity penalties.

---
## Part 3: Holdout Predictions

**Steps:**
1. **3a** — Retrain final GBT pipeline on full `df` (900,188 rows) with best params (`maxDepth=5, maxIter=20`)
2. **3b** — Load `holdout_test.csv`; tag rows with `row_id` before joins
3. **3c** — Join `users.dat` / `movies.dat`; apply same 12-feature engineering using pre-computed `movie_stats` / `user_stats` from training (no leakage); fill nulls with `global_avg`
4. **3d** — Null audit on all 12 feature columns
5. **3e** — Transform → cast `prediction` to `int` → sort by `row_id` → write `predictions.csv` (100,021 rows)

In [28]:
# ── 3a: Retrain final pipeline on ALL 900K training rows ──────────────────────
# Best hyperparameters from TrainValidationSplit: maxDepth=5, maxIter=20
final_gbt = GBTClassifier(
    featuresCol='features',
    labelCol='high_rating',
    maxDepth=best_depth,   # 5
    maxIter=best_iters,    # 20
    seed=42,
)
final_pipeline = Pipeline(stages=[assembler, final_gbt])

print(f'Training on full df : {df.count():,} rows  (maxDepth={best_depth}, maxIter={best_iters})')
print('This takes a few minutes — each of the 20 trees is fit on 900K rows...')
final_model = final_pipeline.fit(df)
print('Retrain complete.')

Training on full df : 900,188 rows  (maxDepth=5, maxIter=20)
This takes a few minutes — each of the 20 trees is fit on 900K rows...
Retrain complete.


In [34]:
# ── 3b: Load holdout_test.csv with Spark ──────────────────────────────────────

HOLDOUT_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=True),
    StructField('MovieID',   IntegerType(), nullable=True),
    StructField('Timestamp', LongType(),    nullable=True),
])
holdout = (
    spark.read
    .option('header', 'true')
    .schema(HOLDOUT_SCHEMA)
    .csv(HOLDOUT_PATH)
)
holdout.printSchema()
holdout.show(3)
print('Holdout loaded.')

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Timestamp: long (nullable = true)

+------+-------+---------+
|UserID|MovieID|Timestamp|
+------+-------+---------+
|  1409|   2011|974763586|
|  2402|   2393|974261582|
|  3551|   1092|966822278|
+------+-------+---------+
only showing top 3 rows

Holdout loaded.


In [35]:
# ── 3c: Join users/movies + replicate training feature engineering ─────────────
h = (
    holdout
    .join(users,  on='UserID',  how='left')
    .join(movies, on='MovieID', how='left')
)

# movie_stats and user_stats were cached from TRAINING data — no leakage
h = h.join(movie_stats, on='MovieID', how='left')
h = h.join(user_stats,  on='UserID',  how='left')

# Fill nulls for movies/users unseen in training
h = h.withColumn('movie_avg_rating',
        F.when(F.col('movie_avg_rating').isNull(), global_avg)
         .otherwise(F.col('movie_avg_rating')))
h = h.withColumn('movie_popularity',
        F.when(F.col('movie_popularity').isNull(), 1)
         .otherwise(F.col('movie_popularity')))
h = h.withColumn('user_avg_rating',
        F.when(F.col('user_avg_rating').isNull(), global_avg)
         .otherwise(F.col('user_avg_rating')))

# Derived features — identical formulas as training
h = h.withColumn('log_movie_popularity', F.log(F.col('movie_popularity') + 1))
h = h.withColumn('release_year',
        F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int'))
h = h.withColumn('movie_age', 2000 - F.col('release_year'))
h = h.withColumn('user_movie_interaction',
        F.col('user_avg_rating') * F.col('movie_avg_rating'))
h = h.withColumn('num_genres', F.size(F.split(F.col('Genres'), r'\|')))
h = h.withColumn('is_action',    F.when(F.col('Genres').contains('Action'),    1).otherwise(0))
h = h.withColumn('is_horror',    F.when(F.col('Genres').contains('Horror'),    1).otherwise(0))
h = h.withColumn('is_war',       F.when(F.col('Genres').contains('War'),       1).otherwise(0))
h = h.withColumn('is_film_noir', F.when(F.col('Genres').contains('Film-Noir'), 1).otherwise(0))
h = h.withColumn('gender_encoded', F.when(F.col('Gender') == 'M', 1).otherwise(0))

# Zero-fill remaining nulls (regex failures on Title, unknown Gender)
h_featured = h.na.fill(0, subset=[
    'release_year', 'movie_age', 'num_genres',
    'is_action', 'is_horror', 'is_war', 'is_film_noir', 'gender_encoded'
]).cache()

n_h = h_featured.count()
print(f'Holdout featured : {n_h:,} rows  |  {len(h_featured.columns)} cols')
assert n_h == 100_021, f'Row count mismatch: {n_h}'

Holdout featured : 100,021 rows  |  23 cols


In [36]:
# ── 3d: Null audit on holdout feature columns ─────────────────────────────────
h_null = h_featured.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in FEATURE_COLS
]).collect()[0].asDict()

total_h_nulls = sum(h_null.values())
if total_h_nulls == 0:
    print('PASS  -- Holdout null audit: 0 nulls across all 12 feature columns.')
else:
    bad = {k: v for k, v in h_null.items() if v > 0}
    print(f'FAIL  -- Nulls remaining: {bad}')
    raise ValueError('Fix null fills above before proceeding to prediction.')

PASS  -- Holdout null audit: 0 nulls across all 12 feature columns.


In [37]:
# ── 3e: Transform → cast → sort → write predictions.csv ──────────────────────
holdout_preds = final_model.transform(h_featured)

holdout_out = (
    holdout_preds
    .withColumn('high_rating_predicted', F.col('prediction').cast('int'))
    .orderBy('UserID', 'MovieID', 'Timestamp')   # stable sort; no row_id needed
    .select('UserID', 'MovieID', 'high_rating_predicted')
)

n_rows = holdout_out.count()
print(f'Holdout prediction rows : {n_rows:,}  (expect 100,021)')
holdout_out.show(5)

out_pd = holdout_out.toPandas()
out_pd.to_csv(OUTPUT_PATH, index=False)
print(f'\nWritten : {OUTPUT_PATH}')

assert len(out_pd) == 100_021, f'FAIL: expected 100,021, got {len(out_pd)}'
print(f'PASS  -- predictions.csv has exactly {len(out_pd):,} rows.')
print(out_pd.head())

Holdout prediction rows : 100,021  (expect 100,021)
+------+-------+---------------------+
|UserID|MovieID|high_rating_predicted|
+------+-------+---------------------+
|     1|   1545|                    1|
|     1|   1907|                    1|
|     1|   2797|                    1|
|     1|   3408|                    1|
|     2|     95|                    0|
+------+-------+---------------------+
only showing top 5 rows


Written : d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\D4\predictions.csv
PASS  -- predictions.csv has exactly 100,021 rows.
   UserID  MovieID  high_rating_predicted
0       1     1545                      1
1       1     1907                      1
2       1     2797                      1
3       1     3408                      1
4       2       95                      0


---
## Part 4: Reflection

### Q1 — What evidence suggests the data has nonlinear patterns that a linear model cannot capture?

The most direct evidence is the **complete reversal of `user_movie_interaction`** between D3 and D4. In D3, Logistic Regression assigned this feature a coefficient of –0.896 — a negative weight — despite it being the strongest Pearson predictor in D2 (|r| = 0.484). This sign flip is a textbook suppressor variable artifact: LR's linear constraint forced it to penalize `user_movie_interaction` to compensate for its overlap with `user_avg_rating` and `movie_avg_rating`. GBT, which evaluates each split independently without collinearity penalties, correctly identified it as the dominant predictor with **83.4% of total feature importance**. No linear model can simultaneously assign a positive weight to `user_avg_rating`, a positive weight to `movie_avg_rating`, and a positive weight to their product — the mathematics of a linear combination make this impossible.

Supporting evidence: despite using identical features and the same training data, GBT improved **AUC-PR from 0.8150 to 0.8191** (+0.0041) and reduced the **FP/FN ratio from 1.76 to 1.65**. These gains come entirely from GBT's ability to express threshold-based interactions (e.g., *if both user and movie average ratings exceed a joint threshold, predict high with very high confidence*) — decision regions that a hyperplane cannot represent.

### Q2 — What would you do differently?

**If we had more time, we would make three changes:**

1. **One-hot encode `primary_genre`.** We currently represent genre with four binary flags (`is_action`, `is_horror`, `is_war`, `is_film_noir`), which captures only a small subset of the 18 genres in the dataset and ignores the primary genre entirely. GBT's feature importance shows genre flags (`is_horror` = 0.3%, `is_action` = 0.06%) are nearly unused — not because genre is unimportant, but because our encoding is too coarse. A one-hot over `primary_genre` (the first genre listed) would give GBT 18 clean binary columns to split on, likely surfacing Drama vs. Comedy vs. Action distinctions that are currently invisible.

2. **Tune `stepSize` (learning rate).** Our grid searched only `maxDepth` and `maxIter`; `stepSize` (the shrinkage factor applied to each tree's contribution) was left at its default of 0.1. A smaller `stepSize` (e.g., 0.05) with more trees typically improves generalization, and is the single most impactful GBT hyperparameter after `maxDepth`. We avoided it to keep the 4-combination grid tractable on 900K rows, but a two-pass search (coarse then fine) would be feasible with more time.

3. **One-hot encode `Age`.** We treat `Age` as an integer (1, 18, 25, 35, 45, 50, 56), but these are category codes — the gap between 1 and 18 is not the same as between 45 and 50. GBT importance shows `Age` = 3.7% (#4 overall), so it carries real signal. Encoding as 7 binary columns would let GBT find sharp boundaries (e.g., the Age=25 cohort rates Action films differently than Age=50) without assuming linear spacing between codes.

---
## Contribution Statement

| Member | Contributions |
|---|---|
| Azatbek Ismailov | |
| Fsehaye Medhanie | |
| Nila Ko | |
| Khaing Min Htwe | |
| Yuexuan Lu | |

*Each team member contributed equally to the design, implementation, and write-up of this deliverable.*